# STRAT-002 Production Backtest

**Strategy:** SOPR + Realized Loss + MVRV Trail

**Framework:** VectorBT for proper backtesting with:
- Realistic execution
- Transaction costs
- Full performance metrics
- Comparison vs benchmark

In [ ]:
# Install vectorbt if needed
# !pip install vectorbt

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import vectorbt as vbt
import warnings
warnings.filterwarnings('ignore')

print(f"VectorBT version: {vbt.__version__}")
print("STRAT-002 Production Backtest 📊")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(mvrv, how='inner').join(sopr, how='inner').join(sopr_sth, how='inner').join(realized_loss, how='inner')
df = df.sort_index()

# Create indicators
df['rl_ma30'] = df['realized_loss'].rolling(30).mean()
df['rl_std30'] = df['realized_loss'].rolling(30).std()
df['rl_zscore'] = (df['realized_loss'] - df['rl_ma30']) / df['rl_std30']

# Filter to test period
df = df[df.index >= '2018-12-15'].dropna()

print(f"Data: {len(df)} rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")

---
## 1. Define Signals

In [ ]:
# STRAT-002 Parameters
RL_Z_THRESHOLD = 0.5
MVRV_TRIGGER = 2.0
TRAIL_PCT = 0.25
STOP_LOSS_PCT = 0.20
MAX_HOLD_DAYS = 9999  # Effectively disabled - test without max hold

# Entry signal: SOPR < 1 AND STH-SOPR < 1 AND RL Z > 0.5
entry_condition = (
    (df['sopr'] < 1) & 
    (df['sopr_sth'] < 1) & 
    (df['rl_zscore'] > RL_Z_THRESHOLD)
)

# First day of signal (not continuation)
entries = entry_condition & ~entry_condition.shift(1).fillna(False)

print(f"Total entry signals: {entries.sum()}")
print(f"\nEntry dates:")
for date in entries[entries].index:
    print(f"  {date.date()}: Price=${df.loc[date, 'price']:,.0f}, MVRV={df.loc[date, 'mvrv']:.2f}")

---
## 2. Custom Exit Logic with VectorBT

VectorBT doesn't have built-in MVRV-triggered trailing stops, so we'll implement custom logic.

In [ ]:
from numba import njit

@njit
def custom_exit_logic(price_arr, mvrv_arr, entry_idx, 
                      mvrv_trigger=2.0, trail_pct=0.25, stop_loss_pct=0.20, max_hold=365):
    """
    Custom exit logic:
    - MVRV > trigger activates trailing stop
    - Trail from peak price
    - Stop loss before trail activates
    - Max hold days
    
    Returns: (exit_idx, exit_price, exit_reason)
    exit_reason: 0=trail, 1=stop_loss, 2=max_hold, 3=end_of_data
    """
    entry_price = price_arr[entry_idx]
    peak_price = entry_price
    trailing_active = False
    
    for j in range(entry_idx + 1, len(price_arr)):
        current_price = price_arr[j]
        current_mvrv = mvrv_arr[j]
        days_held = j - entry_idx
        
        # Update peak
        if current_price > peak_price:
            peak_price = current_price
        
        pnl = (current_price - entry_price) / entry_price
        
        # Check MVRV trigger
        if not trailing_active and current_mvrv >= mvrv_trigger:
            trailing_active = True
        
        # Trailing stop
        if trailing_active:
            trail_stop = peak_price * (1 - trail_pct)
            if current_price <= trail_stop:
                return j, trail_stop, 0  # trail exit
        
        # Stop loss (before trail activates)
        if not trailing_active and pnl <= -stop_loss_pct:
            return j, entry_price * (1 - stop_loss_pct), 1  # stop loss
        
        # Max hold
        if days_held >= max_hold:
            return j, current_price, 2  # max hold
    
    # End of data
    return len(price_arr) - 1, price_arr[-1], 3

In [ ]:
def run_backtest(df, entries, initial_capital=100000, fees=0.001):
    """
    Run full backtest with custom exit logic.
    
    Args:
        df: DataFrame with price, mvrv, etc.
        entries: Boolean series of entry signals
        initial_capital: Starting capital
        fees: Trading fees (0.1% = 0.001)
    """
    price_arr = df['price'].values
    mvrv_arr = df['mvrv'].values
    dates = df.index
    
    entry_indices = np.where(entries.values)[0]
    
    trades = []
    i = 0
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        
        # Get exit
        exit_idx, exit_price, exit_reason = custom_exit_logic(
            price_arr, mvrv_arr, entry_idx,
            MVRV_TRIGGER, TRAIL_PCT, STOP_LOSS_PCT, MAX_HOLD_DAYS
        )
        
        entry_price = price_arr[entry_idx]
        
        # Calculate returns with fees
        gross_return = (exit_price / entry_price) - 1
        net_return = gross_return - (2 * fees)  # Entry + exit fees
        
        exit_reason_map = {0: 'mvrv_trail', 1: 'stop_loss', 2: 'max_hold', 3: 'end_of_data'}
        
        trades.append({
            'entry_date': dates[entry_idx],
            'exit_date': dates[exit_idx],
            'entry_price': entry_price,
            'exit_price': exit_price,
            'gross_return': gross_return,
            'net_return': net_return,
            'days_held': exit_idx - entry_idx,
            'exit_reason': exit_reason_map[exit_reason]
        })
        
        # Skip entries during this trade
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    trades_df = pd.DataFrame(trades)
    
    # Calculate equity curve
    equity = [initial_capital]
    for _, trade in trades_df.iterrows():
        equity.append(equity[-1] * (1 + trade['net_return']))
    
    trades_df['equity_after'] = equity[1:]
    
    return trades_df

In [ ]:
# Run backtest
INITIAL_CAPITAL = 100000
FEES = 0.001  # 0.1% per trade

trades = run_backtest(df, entries, INITIAL_CAPITAL, FEES)

print("TRADE LOG")
print("="*120)
print(f"{'Entry':<12} {'Exit':<12} {'Entry $':>10} {'Exit $':>10} {'Gross':>8} {'Net':>8} {'Days':>6} {'Exit Reason':<12} {'Equity':>12}")
print("-"*120)

for _, t in trades.iterrows():
    print(f"{str(t['entry_date'].date()):<12} {str(t['exit_date'].date()):<12} "
          f"{t['entry_price']:>10,.0f} {t['exit_price']:>10,.0f} "
          f"{t['gross_return']*100:>+7.0f}% {t['net_return']*100:>+7.0f}% "
          f"{t['days_held']:>6} {t['exit_reason']:<12} ${t['equity_after']:>11,.0f}")

---
## 3. Performance Metrics

In [ ]:
def calculate_metrics(trades, initial_capital, df):
    """Calculate comprehensive performance metrics."""
    
    if len(trades) == 0:
        return {}
    
    # Basic stats
    total_trades = len(trades)
    winners = trades[trades['net_return'] > 0]
    losers = trades[trades['net_return'] <= 0]
    
    win_rate = len(winners) / total_trades
    
    # Returns
    total_return = (trades['equity_after'].iloc[-1] / initial_capital) - 1
    
    # CAGR
    start_date = trades['entry_date'].iloc[0]
    end_date = trades['exit_date'].iloc[-1]
    years = (end_date - start_date).days / 365.25
    cagr = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0
    
    # Average trade
    avg_return = trades['net_return'].mean()
    avg_winner = winners['net_return'].mean() if len(winners) > 0 else 0
    avg_loser = losers['net_return'].mean() if len(losers) > 0 else 0
    
    # Profit factor
    gross_profit = winners['net_return'].sum() if len(winners) > 0 else 0
    gross_loss = abs(losers['net_return'].sum()) if len(losers) > 0 else 0.0001
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
    
    # Max drawdown (trade-level)
    equity_curve = [initial_capital] + list(trades['equity_after'])
    peak = equity_curve[0]
    max_dd = 0
    for eq in equity_curve:
        if eq > peak:
            peak = eq
        dd = (eq - peak) / peak
        if dd < max_dd:
            max_dd = dd
    
    # Sharpe (annualized, assuming ~4 trades per year)
    returns = trades['net_return'].values
    if len(returns) > 1:
        sharpe = (returns.mean() / returns.std()) * np.sqrt(4)  # ~4 trades/year
    else:
        sharpe = 0
    
    # Sortino (downside deviation)
    downside_returns = returns[returns < 0]
    if len(downside_returns) > 0:
        downside_std = np.std(downside_returns)
        sortino = (returns.mean() / downside_std) * np.sqrt(4) if downside_std > 0 else float('inf')
    else:
        sortino = float('inf')
    
    # Benchmark comparison (buy & hold)
    bh_start = df.loc[trades['entry_date'].iloc[0], 'price']
    bh_end = df.loc[trades['exit_date'].iloc[-1], 'price']
    bh_return = (bh_end / bh_start) - 1
    bh_cagr = (1 + bh_return) ** (1 / years) - 1 if years > 0 else 0
    
    # Average days in trade
    avg_days = trades['days_held'].mean()
    
    # Exit reason breakdown
    exit_reasons = trades['exit_reason'].value_counts().to_dict()
    
    return {
        'total_trades': total_trades,
        'winners': len(winners),
        'losers': len(losers),
        'win_rate': win_rate,
        'total_return': total_return,
        'cagr': cagr,
        'avg_return': avg_return,
        'avg_winner': avg_winner,
        'avg_loser': avg_loser,
        'profit_factor': profit_factor,
        'max_drawdown': max_dd,
        'sharpe': sharpe,
        'sortino': sortino,
        'avg_days_held': avg_days,
        'bh_return': bh_return,
        'bh_cagr': bh_cagr,
        'alpha': cagr - bh_cagr,
        'exit_reasons': exit_reasons,
        'start_date': start_date,
        'end_date': end_date,
        'years': years,
        'final_equity': trades['equity_after'].iloc[-1]
    }

In [ ]:
metrics = calculate_metrics(trades, INITIAL_CAPITAL, df)

print("\n" + "="*80)
print("PERFORMANCE REPORT: STRAT-002")
print("="*80)

print(f"\n📅 PERIOD")
print(f"   Start: {metrics['start_date'].date()}")
print(f"   End: {metrics['end_date'].date()}")
print(f"   Duration: {metrics['years']:.1f} years")

print(f"\n💰 RETURNS")
print(f"   Total Return: {metrics['total_return']*100:+.0f}%")
print(f"   CAGR: {metrics['cagr']*100:+.1f}%")
print(f"   Buy & Hold Return: {metrics['bh_return']*100:+.0f}%")
print(f"   Buy & Hold CAGR: {metrics['bh_cagr']*100:+.1f}%")
print(f"   Alpha (CAGR - BH): {metrics['alpha']*100:+.1f}%")

print(f"\n📊 TRADE STATISTICS")
print(f"   Total Trades: {metrics['total_trades']}")
print(f"   Winners: {metrics['winners']} ({metrics['win_rate']*100:.0f}%)")
print(f"   Losers: {metrics['losers']} ({(1-metrics['win_rate'])*100:.0f}%)")
print(f"   Avg Trade: {metrics['avg_return']*100:+.1f}%")
print(f"   Avg Winner: {metrics['avg_winner']*100:+.1f}%")
print(f"   Avg Loser: {metrics['avg_loser']*100:.1f}%")
print(f"   Avg Days Held: {metrics['avg_days_held']:.0f}")

print(f"\n📈 RISK METRICS")
print(f"   Profit Factor: {metrics['profit_factor']:.2f}")
print(f"   Max Drawdown: {metrics['max_drawdown']*100:.1f}%")
print(f"   Sharpe Ratio: {metrics['sharpe']:.2f}")
print(f"   Sortino Ratio: {metrics['sortino']:.2f}")

print(f"\n🚪 EXIT REASONS")
for reason, count in metrics['exit_reasons'].items():
    print(f"   {reason}: {count}")

print(f"\n💵 CAPITAL")
print(f"   Initial: ${INITIAL_CAPITAL:,.0f}")
print(f"   Final: ${metrics['final_equity']:,.0f}")

print("\n" + "="*80)

---
## 4. Visualization

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create equity curve with trade markers
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.5, 0.25, 0.25],
                    subplot_titles=['Equity Curve vs Buy & Hold', 'BTC Price with Trades', 'Drawdown'])

# Equity curve
equity_dates = [trades['entry_date'].iloc[0]] + list(trades['exit_date'])
equity_values = [INITIAL_CAPITAL] + list(trades['equity_after'])

fig.add_trace(go.Scatter(
    x=equity_dates, y=equity_values,
    name='Strategy', line=dict(color='green', width=2)
), row=1, col=1)

# Buy & hold equity
bh_start_price = df.loc[trades['entry_date'].iloc[0], 'price']
bh_equity = INITIAL_CAPITAL * (df['price'] / bh_start_price)
fig.add_trace(go.Scatter(
    x=df.index, y=bh_equity,
    name='Buy & Hold', line=dict(color='gray', width=1, dash='dot')
), row=1, col=1)

# Price with trade markers
fig.add_trace(go.Scatter(
    x=df.index, y=df['price'],
    name='BTC Price', line=dict(color='orange', width=1)
), row=2, col=1)

# Entry markers
fig.add_trace(go.Scatter(
    x=trades['entry_date'], y=trades['entry_price'],
    mode='markers', name='Entry',
    marker=dict(color='green', size=10, symbol='triangle-up')
), row=2, col=1)

# Exit markers (color by reason)
colors = {'mvrv_trail': 'blue', 'stop_loss': 'red', 'max_hold': 'purple', 'end_of_data': 'gray'}
for reason in trades['exit_reason'].unique():
    mask = trades['exit_reason'] == reason
    fig.add_trace(go.Scatter(
        x=trades.loc[mask, 'exit_date'], y=trades.loc[mask, 'exit_price'],
        mode='markers', name=f'Exit ({reason})',
        marker=dict(color=colors.get(reason, 'gray'), size=10, symbol='triangle-down')
    ), row=2, col=1)

# Drawdown
equity_series = pd.Series(equity_values, index=equity_dates)
rolling_max = equity_series.expanding().max()
drawdown = (equity_series - rolling_max) / rolling_max

fig.add_trace(go.Scatter(
    x=drawdown.index, y=drawdown * 100,
    fill='tozeroy', name='Drawdown',
    line=dict(color='red')
), row=3, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_yaxes(type='log', row=2, col=1)
fig.update_yaxes(title='Drawdown %', row=3, col=1)

fig.update_layout(height=900, title_text='STRAT-002 Backtest Results')
fig.show()

In [ ]:
# Trade return distribution
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=trades['net_return'] * 100,
    nbinsx=20,
    name='Trade Returns',
    marker_color=['green' if x > 0 else 'red' for x in trades['net_return']]
))

fig.add_vline(x=0, line_dash='dash', line_color='black')
fig.add_vline(x=trades['net_return'].mean()*100, line_dash='dash', line_color='blue', 
              annotation_text=f"Mean: {trades['net_return'].mean()*100:+.1f}%")

fig.update_layout(
    title='Trade Return Distribution',
    xaxis_title='Return %',
    yaxis_title='Count',
    height=400
)
fig.show()

---
## 5. Comparison with VectorBT Built-in (Sanity Check)

In [ ]:
# Use VectorBT's Portfolio for sanity check
# We'll use a simplified version without the MVRV-triggered trail

close = df['price']

# Simple trailing stop version for comparison
pf_simple = vbt.Portfolio.from_signals(
    close,
    entries=entries,
    sl_stop=STOP_LOSS_PCT,
    tp_stop=None,  # No take profit
    fees=FEES,
    init_cash=INITIAL_CAPITAL,
    freq='D'
)

print("\nVectorBT Sanity Check (Simple Stop-Loss Only)")
print("="*60)
print(pf_simple.stats())

---
## 6. Summary Report

In [ ]:
print("\n" + "="*80)
print("STRAT-002 BACKTEST SUMMARY")
print("="*80)

print(f"""
STRATEGY: SOPR + Realized Loss + MVRV Trail

ENTRY RULES:
  - SOPR < 1 (market selling at loss)
  - AND STH-SOPR < 1 (short-term holders selling at loss)
  - AND Realized Loss Z-Score > {RL_Z_THRESHOLD} (above average losses)

EXIT RULES:
  - MVRV > {MVRV_TRIGGER} triggers {TRAIL_PCT*100:.0f}% trailing stop from peak
  - OR {STOP_LOSS_PCT*100:.0f}% stop loss
  - OR {MAX_HOLD_DAYS} days max hold

PERFORMANCE ({metrics['start_date'].date()} to {metrics['end_date'].date()}):
  
  Strategy Return:    {metrics['total_return']*100:+.0f}%
  Buy & Hold Return:  {metrics['bh_return']*100:+.0f}%
  
  Strategy CAGR:      {metrics['cagr']*100:+.1f}%
  Buy & Hold CAGR:    {metrics['bh_cagr']*100:+.1f}%
  Alpha:              {metrics['alpha']*100:+.1f}%
  
  Win Rate:           {metrics['win_rate']*100:.0f}%
  Profit Factor:      {metrics['profit_factor']:.2f}
  Sharpe Ratio:       {metrics['sharpe']:.2f}
  Max Drawdown:       {metrics['max_drawdown']*100:.1f}%
  
  ${INITIAL_CAPITAL:,} → ${metrics['final_equity']:,.0f}
""")

print("="*80)

In [ ]:
# Save results
import json

# Convert timestamps to strings for JSON
trades_export = trades.copy()
trades_export['entry_date'] = trades_export['entry_date'].astype(str)
trades_export['exit_date'] = trades_export['exit_date'].astype(str)

results = {
    'strategy': 'STRAT-002',
    'parameters': {
        'rl_z_threshold': RL_Z_THRESHOLD,
        'mvrv_trigger': MVRV_TRIGGER,
        'trail_pct': TRAIL_PCT,
        'stop_loss_pct': STOP_LOSS_PCT,
        'max_hold_days': MAX_HOLD_DAYS,
        'initial_capital': INITIAL_CAPITAL,
        'fees': FEES
    },
    'metrics': {k: float(v) if isinstance(v, (np.floating, np.integer)) else v 
                for k, v in metrics.items() if k not in ['exit_reasons', 'start_date', 'end_date']},
    'exit_reasons': metrics['exit_reasons'],
    'trades': trades_export.to_dict('records')
}

# Handle dates
results['metrics']['start_date'] = str(metrics['start_date'].date())
results['metrics']['end_date'] = str(metrics['end_date'].date())

with open('../data/strat002_backtest_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

print("Results saved to ../data/strat002_backtest_results.json")